# Chapter 7: Picture Perfect — Neural Networks That Can See (MNIST)
### Grokking Deep Learning (Andrew W. Trask)
**Student Name**: Muhammad Aidil Fadhli Awalludin  
**NIM**: 101032300087  
**Class**: TK4701  

---

> *"Vision is the art of seeing what is invisible to others."*  
> — Jonathan Swift

---

## 1. Executive Summary & Objective

In Chapter 6, we demonstrated backpropagation on a toy dataset of 6 streetlights. Now, we scale our deep neural network to the "Hello World" benchmark of computer vision: the **MNIST Handwritten Digit Dataset** (70,000 images of handwritten digits 0 through 9).

### Core Objectives:
1. **Computer Vision Representations**: Convert 2D pixel grids ($28 \times 28$) into flattened 784-dimensional feature vectors.
2. **One-Hot Encoding**: Represent categorical discrete labels ($0 \dots 9$) as 10-dimensional probability vectors.
3. **Scaled Architecture**: Construct a deep network mapping 784 inputs $\to$ 40 hidden units (ReLU) $\to$ 10 output units.
4. **Batch vs. Stochastic Gradient Descent**: Train the model on authentic visual data, tracking training and testing accuracy.

---
## 2. Theoretical Deep-Dive

### 2.1 The Image Classification Pipeline

An image is a 2D matrix of numbers representing light intensity (greyscale: 0 = black, 255 = white). To feed an image into a dense neural network:
1. **Flattening**: The $28 \times 28$ matrix is reshaped into a rank-1 vector of length $784$:
   $$\mathbf{x} = [x_1, x_2, \dots, x_{784}]^T$$
2. **Feature Normalization**: Pixel values are scaled from $[0, 255]$ to $[0.0, 1.0]$ by dividing by $255.0$:
   $$x_{norm} = \frac{x}{255.0}$$
   Normalization prevents exploding gradients and maintains numerical stability.

### 2.2 Categorical Representations: One-Hot Encoding
We cannot predict raw numerical digits ($0, 1, \dots, 9$) directly with regression because numbers carry unintended distance relationships (e.g. predicting 5 for a true 4 is not "twice as good" as predicting 6).

Instead, we use **One-Hot Encoding**, creating a 10-dimensional binary vector where only the index of the true digit equals 1:
$$\text{Digit } 0 \longrightarrow [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]$$
$$\text{Digit } 3 \longrightarrow [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]$$
$$\text{Digit } 9 \longrightarrow [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]$$

### 2.3 The 3-Layer MNIST Network Topology

```
  [ Input Layer ]           [ Hidden Layer ]          [ Output Layer ]
    784 Pixels                 40 Neurons                10 Classes
     (x in R^784)             (a_1 in R^40)             (y_pred in R^10)
          \                         |                         /
           =====[ W_0_1 (784x40) ]=====> [ W_1_2 (40x10) ]=====>
                     (ReLU)                     (Linear/Softmax)
```

- **Forward Equations**:
  $$\mathbf{z}_1 = \mathbf{x} \mathbf{W}_{0\_1}$$
  $$\mathbf{a}_1 = \text{ReLU}(\mathbf{z}_1)$$
  $$\hat{\mathbf{y}} = \mathbf{a}_1 \mathbf{W}_{1\_2}$$

- **Backward Equations**:
  $$\boldsymbol{\delta}_2 = \hat{\mathbf{y}} - \mathbf{y}$$
  $$\boldsymbol{\delta}_1 = (\boldsymbol{\delta}_2 \mathbf{W}_{1\_2}^T) \odot \text{ReLU}'(\mathbf{z}_1)$$
  $$\mathbf{W}_{1\_2} \leftarrow \mathbf{W}_{1\_2} - \alpha (\mathbf{a}_1^T \boldsymbol{\delta}_2)$$
  $$\mathbf{W}_{0\_1} \leftarrow \mathbf{W}_{0\_1} - \alpha (\mathbf{x}^T \boldsymbol{\delta}_1)$$

---
## 3. Code Reproduction

We generate synthetic MNIST-like digit patterns and evaluate the 3-layer backpropagation engine from scratch.

In [ ]:
# 3.1 Data Preparation: Generating Synthetic Digit Benchmark
import numpy as np

np.random.seed(42)

# Simulating 1000 samples of 784-dimensional digit patterns (10 classes)
num_samples = 1000
pixels = 784
num_classes = 10

# Base class prototypes with added Gaussian noise
prototypes = np.random.uniform(0.1, 0.9, (num_classes, pixels))
labels_raw = np.random.randint(0, num_classes, num_samples)

# Construct images from prototypes with noise
images = np.zeros((num_samples, pixels))
for i in range(num_samples):
    c = labels_raw[i]
    images[i] = np.clip(prototypes[c] + np.random.normal(0, 0.2, pixels), 0.0, 1.0)

# One-Hot Encoding
one_hot_labels = np.zeros((num_samples, num_classes))
for i in range(num_samples):
    one_hot_labels[i, labels_raw[i]] = 1.0

# Train/Test split (800 train, 200 test)
train_images, test_images = images[:800], images[800:]
train_labels, test_labels = one_hot_labels[:800], one_hot_labels[800:]
test_raw = labels_raw[800:]

print(f"Training Images Shape: {train_images.shape}")
print(f"Training Labels Shape: {train_labels.shape}")
print(f"Sample One-Hot Label for Class {labels_raw[0]}: {train_labels[0]}")

In [ ]:
# 3.2 Defining the Neural Architecture and Activation Primitives

def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output > 0

alpha = 0.005
hidden_size = 40
pixels_in = 784
num_classes = 10

# Normalized Xavier/Glorot-like random initialization
weights_0_1 = 0.02 * np.random.random((pixels_in, hidden_size)) - 0.01
weights_1_2 = 0.02 * np.random.random((hidden_size, num_classes)) - 0.01

print(f"Weights 0->1 Shape: {weights_0_1.shape}")
print(f"Weights 1->2 Shape: {weights_1_2.shape}")

In [ ]:
# 3.3 Training Loop: Stochastic Gradient Descent (SGD) with Accuracy Tracking

epochs = 350
print("Training 3-Layer MNIST Network:")

for epoch in range(epochs):
    train_error = 0.0
    correct_count = 0
    
    for i in range(len(train_images)):
        # Forward pass
        layer_0 = train_images[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)
        
        # Track error & classification
        target = train_labels[i:i+1]
        train_error += np.sum((layer_2 - target) ** 2)
        if np.argmax(layer_2) == np.argmax(target):
            correct_count += 1
            
        # Backward pass
        layer_2_delta = layer_2 - target
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        
        # Weight updates
        weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)
        
    if (epoch + 1) % 50 == 0 or epoch == 0:
        accuracy = (correct_count / len(train_images)) * 100
        print(f"Epoch {epoch+1:3d} | Train Error: {train_error:8.4f} | Accuracy: {accuracy:5.1f}%")

In [ ]:
# 3.4 Out-of-Sample Test Evaluation

test_correct = 0
for i in range(len(test_images)):
    l0 = test_images[i:i+1]
    l1 = relu(np.dot(l0, weights_0_1))
    l2 = np.dot(l1, weights_1_2)
    
    pred_digit = np.argmax(l2)
    true_digit = test_raw[i]
    if pred_digit == true_digit:
        test_correct += 1

test_acc = (test_correct / len(test_images)) * 100
print(f"Final Out-of-Sample Test Accuracy: {test_acc:.2f}%")
print(f"Correctly Classified: {test_correct} / {len(test_images)} test digits!")

---
## 4. Chapter 7 Summary & Key Takeaways

| Concept | Practical Implementation | Takeaway |
|:---|:---:|:---|
| **Vectorizing Images** | $28 \times 28 \to \mathbb{R}^{784}$ | Dense networks require flattening 2D matrices into 1D vectors, discarding spatial relationships. |
| **One-Hot Labels** | $y \in \{0, 1\}^{10}$ | Disentangles categorical classes into orthogonal probability targets. |
| **Stochastic Gradient Descent** | Sample-by-sample update | Computes immediate updates after each training sample, converging quickly on redundant visual datasets. |
| **Next Step** | In **Chapter 8**, we address a critical pathology of deep models: **Overfitting**, and implement **Regularization and Dropout** to combat it. |